# 02 Data Cleaning

In this notebook, we clean and prepare the Formula 1 dataset collected in the previous step.

The main goals are:

- load raw race results
- load raw qualifying results
- check dataset structure
- convert columns to correct data types
- merge race and qualifying data
- create useful basic columns
- handle obvious missing values
- save a clean dataset for feature engineering and machine learning

# Import libs

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set Project Paths

In [ ]:
PROJECT_ROOT = Path("..")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Load Raw Dataset

In [ ]:
race_results_path = RAW_DATA_DIR / "race_results_2018_2026.csv"
qualifying_path = RAW_DATA_DIR / "qualifying_results_2018_2026.csv"
schedule_path = RAW_DATA_DIR / "race_schedule_2018_2026.csv"

race_results_df = pd.read_csv(race_results_path)
qualifying_df = pd.read_csv(qualifying_path)
schedule_df = pd.read_csv(schedule_path)

In [ ]:
race_results_df.head()

In [ ]:
qualifying_df.head()

In [ ]:
schedule_df.head()

# Check Dataset shapes

In [ ]:
"Race results shape: ", race_results_df.shape

In [ ]:
"Qualifying results shape: ", qualifying_df.shape

In [ ]:
"Schedule shape: ", schedule_df.shape

# Check columns

In [ ]:
race_results_df.columns

In [ ]:
qualifying_df.columns

In [ ]:
schedule_df.columns

# Convert Numeric Columns

In [ ]:
race_num_col = [
    "season",
    "round",
    "grid",
    "position",
    "points",
    "laps"
]

for col in race_num_col:
    race_results_df[col] = pd.to_numeric(race_results_df[col], errors="coerce")

In [ ]:
qual_num_col = [
    "season",
    "round",
    "qualifying_position"
]

for col in qual_num_col:
    qualifying_df[col] = pd.to_numeric(qualifying_df[col], errors="coerce")

In [ ]:
race_results_df.dtypes

In [ ]:
qualifying_df.dtypes

# Create Driver full name

create a readable driver name column for analysis, visualization, and driver cards

In [ ]:
race_results_df['driver_name'] = (race_results_df['driver_given_name'] + " " + race_results_df['driver_family_name'])

In [ ]:
race_results_df[['driver_id', "driver_name"]].drop_duplicates().head()

# Select useful Race col

In [ ]:
race_columns = [
    "season",
    "round",
    "race_name",
    "race_date",
    "circuit_id",
    "circuit_name",
    "circuit_country",
    "circuit_locality",
    
    "driver_id",
    "driver_name",
    "driver_code",
    "driver_nationality",
    
    "constructor_id",
    "constructor_name",
    "constructor_nationality",
    
    "grid",
    "position",
    "position_text",
    "points",
    "laps",
    "status"
]

race_clean = race_results_df[race_columns].copy()

race_clean.head()

# Select useful Qual col

In [ ]:
qualifying_columns = [
    "season",
    "round",
    "driver_id",
    "qualifying_position",
    "q1",
    "q2",
    "q3"
]

qualifying_clean = qualifying_df[qualifying_columns].copy()

qualifying_clean.head()

# Merge Race and qual results

In [ ]:
f1_clean_df = race_clean.merge(qualifying_clean, on = ["season", "round", "driver_id"], how="left")

f1_clean_df.head()

In [ ]:
f1_clean_df.shape

# Check missing values

In [ ]:
f1_clean_df.isnull().sum().sort_values(ascending=False)

In [ ]:
f1_clean_df[["grid", "qualifying_position", "position", "points", "status"]].isnull().sum()

# Handle with missing values

If qualifying position is missing, we use grid position as a fallback.

In [ ]:
f1_clean_df['qualifying_position'] = f1_clean_df['qualifying_position'].fillna(f1_clean_df['grid'])

In [ ]:
f1_clean_df[["qualifying_position"]].isnull().sum()

# Create DNF feature

In [ ]:
finish_status = [
    "Finished",
    "+1 Lap",
    "+2 Laps",
    "+3 Laps",
    "+4 Laps",
    "+5 Laps",
    "+6 Laps",
]

f1_clean_df["dnf"] = np.where(f1_clean_df['status'].isin(finish_status), 0, 1)

f1_clean_df[["driver_name", "race_name", "status", "dnf"]].head(20)

# Create Point Finish Feature

wil show if driver achive point

In [ ]:
f1_clean_df["points_finish"] = np.where(f1_clean_df["points"] > 0, 1, 0)

f1_clean_df[["driver_name", "race_name", "position", "points", "points_finish"]].head()

# Create Podium Feature

In [ ]:
f1_clean_df["podium"] = np.where(f1_clean_df["position"] <= 3, 1, 0)

f1_clean_df[["driver_name", "race_name", "position", "podium"]].head()

# Fix Grid Position Value 0

In Formula 1 data, grid position 0 usually means the driver started from the pit lane or had no standard grid position.

For the first version, we replace 0 with the maximum grid position in that race.

In [ ]:
def fix_grid_pos(group):
    max_grid = group["grid"].max()
    group["grid"] = group["grid"].replace(0, max_grid)
    return group

f1_clean_df = (f1_clean_df
               .groupby(["season", "round"], group_keys=False)
               .apply(fix_grid_pos)
               )

In [ ]:
f1_clean_df[f1_clean_df["grid"] == 0]

# Check Duplicates

In [ ]:
duplicates = f1_clean_df.duplicated(subset=["season", "round", "driver_id"], keep=False)

f1_clean_df[duplicates]

# Sort dataset

In [ ]:
f1_clean_df =  f1_clean_df.sort_values(by=["season", "round", "position"]).reset_index(drop=True)

f1_clean_df.head()

# Final check

In [ ]:
f1_clean_df.info()

In [ ]:
f1_clean_df.head(20)

In [ ]:
f1_clean_df.isnull().sum().sort_values(ascending = False)

# Save clean dataset

In [ ]:
clean_set_path = PROCESSED_DATA_DIR / "f1_clean_2018_2026.csv"

f1_clean_df.to_csv(clean_set_path, index= False)

clean_set_path

# Reload saved dataset

In [ ]:
f1_clean_check = pd.read_csv(clean_set_path)

f1_clean_check.head()

In [ ]:
f1_clean_check.shape